# LangChain v1 快速入门：接入在线大模型与构建 AI 应用链路

> **本笔记基于 LangChain v1.x（最新稳定版）**编写，对照旧版 v0.x 的 API 变化，帮助你快速从 v0.2 迁移到 v1。  
> 原版笔记：[LangChain v0.2 学习笔记](./LangChain.ipynb)

---

## 0. 引言：什么是 LangChain v1？

2025年10月，LangChain 正式发布 **v1.0**。这是一次生产级大规模重构，核心围绕三个方向：

| 变化 | 说明 |
|------|------|
| **`create_agent`** | 全新 Agent 构建标准，替代旧的 `create_react_agent` |
| **Standard Content Blocks** | 跨供应商统一消息内容表示 `content_blocks` |
| **Simplified Package** | 精简包命名空间，旧版 Chains 等移至 `langchain-classic` |

### 0.1 v1 vs v0.x：关键 API 变化速查表

| 功能 | v0.x（旧） | v1.x（新） |
|------|-----------|-----------|
| 模型初始化 | `ChatOpenAI(model="gpt-4")` | `init_chat_model("openai:gpt-4o")` |
| 简单链 | `LLMChain(llm=..., prompt=...)` | LCEL：`prompt \| model \| parser` |
| Agent | `create_react_agent(...)` | `create_agent(...)` |
| 消息导入 | `from langchain_core.messages import ...` | `from langchain.messages import ...`（推荐） |
| 工具装饰器 | `from langchain_core.tools import tool` | `from langchain.tools import tool` |
| Python 版本 | ≥ 3.9 | **≥ 3.10** |

> ⚠️ **重要**：v1 不再内置 `LLMChain`、`ConversationChain`、`MultiQueryRetriever` 等旧式组件。如需使用，请安装 `langchain-classic` 包。本笔记将完全使用 v1 原生 API。

## 1. 环境安装

### 1.1 基础依赖

In [ ]:
# 安装 LangChain v1 核心包（会自动安装 langchain-core）
!pip install langchain>=1.0

# OpenAI 集成包（推荐）
!pip install langchain-openai

# 可选：其他模型提供商
# !pip install langchain-anthropic    # Anthropic Claude
# !pip install langchain-google-genai # Google Gemini
# !pip install langchain-ollama      # 本地 Ollama 模型
# !pip install langchain-deepseek    # DeepSeek

### 1.2 检查版本

In [ ]:
import langchain
print(f"LangChain version: {langchain.__version__}")
# 应输出 >= 1.0.0

### 1.3 设置 API Key（以 OpenAI 为例）

In [ ]:
import os

# 方式一：直接设置环境变量（推荐在实际项目中使用）
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"

# 方式二：从 .env 文件加载
# from dotenv import load_dotenv
# load_dotenv()

# 方式三：在代码中直接传入（仅用于快速测试，切勿提交到 Git!）
# model = init_chat_model("openai:gpt-4o-mini", api_key="sk-...")

print("✅ 环境准备就绪，请确保设置了有效的 API Key")

---

## 2. Model I/O：模型输入输出的三要素

LangChain 将模型交互抽象为三个标准化环节：

```
用户输入 → [Format] → [Predict] → [Parse] → 结构化输出
          Prompt       Model      Output
          Template                 Parser
```

- **Format（格式化）**：将用户输入 + 模板 → 模型可理解的消息
- **Predict（预测）**：消息 → 模型推理 → 响应消息
- **Parse（解析）**：响应消息 → 解析为结构化数据

### 2.1 Format：Prompt Templates（提示词模板）

#### a) PromptTemplate — 简单字符串模板

In [ ]:
from langchain_core.prompts import PromptTemplate

# 定义一个简单的模板
template = PromptTemplate.from_template(
    "请用{language}语言，写一首关于{topic}的简短诗歌。"
)

# 用变量填充模板
formatted = template.format(language="中文", topic="春天")
print(formatted)

In [ ]:
# PromptTemplate 也支持 LCEL 管道
prompt = PromptTemplate.from_template("请用3句话介绍：{topic}")
prompt_value = prompt.invoke({"topic": "LangChain v1"})

print("类型:", type(prompt_value))
print("内容:", prompt_value.to_string())

#### b) ChatPromptTemplate — 多角色消息模板

这是构建对话应用的**推荐方式**，可以定义 System / Human / AI 等多角色消息。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.messages import HumanMessage, SystemMessage, AIMessage

# 定义多角色模板
chat_template = ChatPromptTemplate.from_messages([
    ("system", "你是一位{role}专家，请用{language}回答问题。"),
    ("human", "{question}"),
])

# 查看模板生成的消息
messages = chat_template.invoke({
    "role": "Python编程",
    "language": "中文",
    "question": "什么是装饰器？"
})

for msg in messages:
    print(f"[{msg.type.upper()}] {msg.content}")
    print("---")

### 2.2 Predict：模型初始化 — `init_chat_model`

v1 推荐使用 **`init_chat_model`** 统一初始化所有聊天模型，一行代码搞定。

**语法**：`init_chat_model("provider:model_name", **kwargs)`

In [ ]:
from langchain.chat_models import init_chat_model

# --- 方式一：固定模型（最常用） ---
model = init_chat_model(
    "openai:gpt-4o-mini",   # 格式："提供商:模型名"
    temperature=0.7,        # 控制随机性 (0.0-2.0)
    max_tokens=512,         # 限制最大输出token数
)

# 调用模型
response = model.invoke("你好，请用一句话介绍你自己。")
print("回复:", response.content)

# 查看 token 用量
print(f"\nToken 用量: {response.usage_metadata}")

In [ ]:
# --- 方式二：可配置模型（运行时切换） ---
# 不预先指定模型，而是在调用时通过 config 动态指定

configurable_model = init_chat_model()  # 默认 ('model', 'model_provider') 可配置

# 运行时指定模型
result = configurable_model.invoke(
    "你是谁？",
    config={"configurable": {"model": "openai:gpt-4o-mini"}}
)
print(result.content[:50], "...")

In [ ]:
# --- 方式三：多模型对比 ---
# 定义两个不同的模型
gpt_model = init_chat_model("openai:gpt-4o-mini", temperature=0.7)
# deepseek_model = init_chat_model("deepseek:deepseek-chat", temperature=0.7)

# 同一条 prompt 发给不同模型
prompt_text = "用一句话解释什么是机器学习。"

gpt_response = gpt_model.invoke(prompt_text)
print(f"[GPT-4o-mini] {gpt_response.content}")

# 如果你有 DeepSeek API Key，取消下面注释：
# ds_response = deepseek_model.invoke(prompt_text)
# print(f"[DeepSeek] {ds_response.content}")

#### 直接使用 ChatOpenAI（兼容旧方式）

`init_chat_model` 内部也是调用各提供商的类。你也可以继续使用传统的直接初始化方式：

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
result = model.invoke("你好！")
print(result.content)

### 2.3 Parse：输出解析器（Output Parsers）

#### a) StrOutputParser — 提取纯文本

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

# 构建链：prompt → model → parser
prompt = ChatPromptTemplate.from_template("请用一句话介绍：{topic}")
model = init_chat_model("openai:gpt-4o-mini", temperature=0.5)
parser = StrOutputParser()

chain = prompt | model | parser

result = chain.invoke({"topic": "量子计算"})
print("结果:", result)
print("类型:", type(result))  # <class 'str'>

#### b) PydanticOutputParser — 解析为结构化对象

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# 1. 定义期望的数据结构
class BookInfo(BaseModel):
    title: str = Field(description="书名")
    author: str = Field(description="作者")
    summary: str = Field(description="一句话简介")
    rating: float = Field(description="评分，1-10分")

# 2. 创建解析器并获取格式说明
parser = PydanticOutputParser(pydantic_object=BookInfo)

# 3. 将格式说明注入到 prompt 中
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个图书信息助手。\n{format_instructions}"),
    ("human", "请介绍一本书：《{book_name}》"),
])

# 写入格式说明
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# 4. 构建链
chain = prompt | model | parser

# 5. 调用
book = chain.invoke({"book_name": "三体"})
print(f"书名: {book.title}")
print(f"作者: {book.author}")
print(f"简介: {book.summary}")
print(f"评分: {book.rating}/10")
print(f"\n解析后类型: {type(book)}")  # <class '__main__.BookInfo'>

---

## 3. 用 LCEL（LangChain Expression Language）构建处理链

> ⚠️ **v1 重要变化**：旧的 `LLMChain`、`ConversationChain` 已被移除。全部使用 **LCEL 语法**：`prompt | model | parser`

LCEL 使用 `|` 管道符连接组件，数据从左到右流经每一个环节。

### 3.1 基础链：prompt → model → parser

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-4o-mini", temperature=0.7)

# 构建一条翻译链
translation_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一位专业翻译，请将用户输入的文本翻译成{target_lang}，只输出译文。"),
    ("human", "{text}"),
])

translation_chain = translation_prompt | model | StrOutputParser()

# 调用
result = translation_chain.invoke({
    "target_lang": "英文",
    "text": "人工智能正在改变我们的生活方式。"
})
print(result)

### 3.2 批量处理（Batch）

In [ ]:
# 批量翻译多个句子
inputs = [
    {"target_lang": "英文", "text": "今天天气真好。"},
    {"target_lang": "英文", "text": "我喜欢学习新技术。"},
    {"target_lang": "英文", "text": "LangChain让开发变简单了。"},
]

results = translation_chain.batch(inputs)
for i, (inp, out) in enumerate(zip(inputs, results)):
    print(f"[{i+1}] {inp['text']} → {out}")

### 3.3 流式输出（Streaming）

In [ ]:
# 流式输出：逐个 token 返回结果
chain = ChatPromptTemplate.from_template("写一首关于{topic}的五言绝句。") | model | StrOutputParser()

print("流式输出：")
for chunk in chain.stream({"topic": "夏天"}):
    print(chunk, end="", flush=True)

print("\n\n✅ 流式输出完成")

### 3.4 链的组合与嵌套

In [ ]:
# LCEL 支持链的组合
# 例如：先翻译，再摘要

summary_prompt = ChatPromptTemplate.from_template("请用一句话总结以下内容：\n{text}")
summary_chain = summary_prompt | model | StrOutputParser()

# 组合链：翻译 → 摘要
combined_chain = translation_chain | summary_chain  # 上一个链的输出是字符串，作为 text 输入

result = combined_chain.invoke({
    "target_lang": "英文",
    "text": "LangChain 是一个用于构建大语言模型应用的框架，它提供了模块化的组件和链式调用能力。"
})
print("组合链结果:", result)

### 3.5 RunnableParallel：并行执行

In [ ]:
from langchain_core.runnables import RunnableParallel

# 同时执行翻译和摘要
translate_and_summarize = RunnableParallel(
    translation=translation_chain,
    summary=summary_chain,
)

result = translate_and_summarize.invoke({
    "target_lang": "英文",
    "text": "人工智能技术正在快速发展，深度学习模型变得越来越强大。"
})

print("翻译:", result["translation"])
print("摘要:", result["summary"])

---

## 4. Callbacks 回调系统

Callbacks 让你能够在 LLM 调用的各个阶段插入自定义逻辑，如日志记录、token 统计、流式输出等。

### 4.1 自定义 Callback Handler

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model
import time

class MyCallback(BaseCallbackHandler):
    """自定义回调处理器：记录每个阶段的耗时"""
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.start_time = time.time()
        print(f"[LLM Start] 发送 {len(prompts)} 条提示，消息数: {len(prompts[0]) if isinstance(prompts[0], list) else 1}")
    
    def on_llm_end(self, response, **kwargs):
        elapsed = time.time() - self.start_time
        print(f"[LLM End] 耗时: {elapsed:.2f}s")
    
    def on_llm_error(self, error, **kwargs):
        print(f"[LLM Error] {error}")
    
    def on_chain_start(self, serialized, inputs, **kwargs):
        print(f"[Chain Start] 链开始执行")
    
    def on_chain_end(self, outputs, **kwargs):
        print(f"[Chain End] 链执行完成")

# 使用回调
model = init_chat_model("openai:gpt-4o-mini", temperature=0.5)
prompt = ChatPromptTemplate.from_template("用一句话介绍：{topic}")
chain = prompt | model | StrOutputParser()

callback = MyCallback()
result = chain.invoke(
    {"topic": "黑洞"},
    config={"callbacks": [callback]}
)
print(f"\n结果: {result}")

### 4.2 流式输出 Callback

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

class StreamingCallback(BaseCallbackHandler):
    """流式输出回调：实时打印生成的每个 token"""
    
    def on_llm_new_token(self, token: str, **kwargs):
        print(token, end="", flush=True)
    
    def on_llm_end(self, response, **kwargs):
        print()  # 换行

# 使用流式回调 + stream
chain = ChatPromptTemplate.from_template("请介绍：{topic}") | model
stream_callback = StreamingCallback()

print("流式输出：")
result = chain.invoke(
    {"topic": "LangChain v1"},
    config={"callbacks": [stream_callback]}
)
print(f"\n完整结果: {result.content[:50]}...")

### 4.3 Token 用量统计（v1 新特性）

In [ ]:
from langchain_core.callbacks import UsageMetadataCallbackHandler

# 创建用量追踪回调
usage_callback = UsageMetadataCallbackHandler()

chain = ChatPromptTemplate.from_template("请介绍：{topic}") | model | StrOutputParser()

# 连续调用，累积统计
for topic in ["Python", "机器学习", "LangChain"]:
    chain.invoke({"topic": topic}, config={"callbacks": [usage_callback]})

# 查看聚合的 token 用量
print("Token 用量汇总:")
for model_name, usage in usage_callback.usage_metadata.items():
    print(f"  模型: {model_name}")
    print(f"    输入 tokens: {usage.get('input_tokens', 'N/A')}")
    print(f"    输出 tokens: {usage.get('output_tokens', 'N/A')}")
    print(f"    总计 tokens: {usage.get('total_tokens', 'N/A')}")

---

## 5. 对话记忆（Memory）

> **v1 变化**：旧的 `ConversationBufferMemory` 已移到 `langchain-classic`。  
> v1 推荐**手动管理消息历史**，对于复杂场景使用 LangGraph 的 Checkpointing。

### 5.1 手动管理消息历史（推荐）

In [ ]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-4o-mini", temperature=0.7)

# 手动维护消息列表
messages = [
    SystemMessage(content="你是一个友好的AI助手，请用中文回答。"),
]

def chat(user_input: str):
    """模拟对话：将历史消息和新输入一起发给模型"""
    messages.append(HumanMessage(content=user_input))
    response = model.invoke(messages)
    messages.append(response)  # 将 AI 回复也加入历史
    return response.content

# 第一轮
print("用户: 你好，我叫小明")
print("AI:", chat("你好，我叫小明"))
print(f"\n历史消息数: {len(messages)}\n")

# 第二轮（能记住之前的名字）
print("用户: 我叫什么名字？")
print("AI:", chat("我叫什么名字？"))
print(f"\n历史消息数: {len(messages)}\n")

# 第三轮
print("用户: 推荐一本编程入门书")
print("AI:", chat("推荐一本编程入门书"))
print(f"\n历史消息数: {len(messages)}")

### 5.2 `trim_messages`：管理上下文窗口

当对话变长，超过模型上下文窗口时，需要用 `trim_messages` 裁剪历史消息。

In [ ]:
from langchain.messages import trim_messages, HumanMessage, SystemMessage, AIMessage

# 模拟一段长对话历史
history = [
    SystemMessage(content="你是一个助手。"),
    HumanMessage(content="问题1"),
    AIMessage(content="回答1"),
    HumanMessage(content="问题2"),
    AIMessage(content="回答2"),
    HumanMessage(content="问题3"),
    AIMessage(content="回答3"),
    HumanMessage(content="问题4"),
    AIMessage(content="回答4"),
    HumanMessage(content="问题5"),
]

# 裁剪：保留最近的消息，总 token 不超过上限
trimmed = trim_messages(
    history,
    max_tokens=50,              # 最大 token 数
    strategy="last",            # 保留最后的（"last"）消息
    token_counter=model,        # 使用同一模型计数
    include_system=True,        # 始终保留 system 消息
    start_on="human",           # 从 human 消息开始（确保对话完整性）
)

print(f"原始消息数: {len(history)}")
print(f"裁剪后: {len(trimmed)}")
for msg in trimmed:
    print(f"  [{msg.type}] {msg.content}")

### 5.3 LangGraph 持久化简介（进阶）

对于复杂的对话场景，v1 推荐使用 **LangGraph** 的 Checkpointing 机制自动管理对话状态。这是 v1 的核心能力之一，能轻松实现：

- 对话历史的自动持久化
- 对话的时间旅行（回溯到任意历史节点）
- 多轮人机协同

> 详见 LangGraph 文档，此处不作展开，仅展示概念。

---

## 6. Bonus：`create_agent` — v1 全新 Agent 构建标准

这是 v1 最核心的变化之一。`create_agent` 替代了旧版的 `create_react_agent`，构建体验更加简洁，并且引入了**中间件（Middleware）**系统。

### 6.1 创建工具（Tools）

v1 中 `@tool` 装饰器从 `langchain.tools` 导入：

In [ ]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气信息"""
    # 实际项目中这里会调用天气 API
    weather_data = {
        "北京": "晴，25°C，湿度40%",
        "上海": "多云，28°C，湿度65%",
        "深圳": "阵雨，30°C，湿度80%",
        "杭州": "阴，22°C，湿度55%",
    }
    return weather_data.get(city, f"未找到{city}的天气数据")

@tool
def calculate(expression: str) -> str:
    """执行数学计算，例如 '2+3*4' """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {e}"

@tool
def get_current_time() -> str:
    """获取当前时间"""
    from datetime import datetime
    return f"当前时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

# 验证工具
print(get_weather.invoke({"city": "北京"}))
print(calculate.invoke({"expression": "15 * 3 + 7"}))
print(get_current_time.invoke({}))

### 6.2 创建 Agent

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 初始化模型
model = init_chat_model("openai:gpt-4o-mini", temperature=0)

# 创建 Agent
agent = create_agent(
    model=model,
    tools=[get_weather, calculate, get_current_time],
    system_prompt="你是一个有用的助手，可以使用工具来回答问题。请用中文回答。",
)

# 查看 Agent 结构
print(f"Agent 类型: {type(agent).__name__}")
print(f"可用工具: {[t.name for t in agent.tools] if hasattr(agent, 'tools') else '查看 agent 内部'}")

### 6.3 运行 Agent

In [ ]:
# 测试 1：天气查询（会调用工具）
response = agent.invoke({
    "messages": [{"role": "user", "content": "北京今天天气怎么样？"}]
})

# Agent 返回的消息列表包含完整的推理过程
for msg in response["messages"]:
    role = getattr(msg, 'type', msg.get('role', 'unknown'))
    content = getattr(msg, 'content', str(msg))
    if role == "ai" and content:
        print(f"[AI] {content}")
    elif role == "tool":
        print(f"[Tool] {getattr(msg, 'name', '')}: {content}")

In [ ]:
# 测试 2：数学计算
response = agent.invoke({
    "messages": [{"role": "user", "content": "帮我算一下 256 * 128 等于多少？"}]
})

# 只提取最终 AI 回复
for msg in response["messages"]:
    if getattr(msg, 'type', '') == "ai" and getattr(msg, 'content', ''):
        print(f"最终回答: {msg.content}")

In [ ]:
# 测试 3：多轮对话（Agent 会自动决定是否需要调用工具）
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "现在几点了？另外深圳今天天气如何？"}
    ]
})

for msg in response["messages"]:
    role = getattr(msg, 'type', msg.get('role', 'unknown'))
    content = getattr(msg, 'content', str(msg))
    if role == "ai" and content:
        print(f"[AI] {content}")

### 6.4 Agent 流式输出

In [ ]:
# Agent 也支持流式输出（底层基于 LangGraph）
for event in agent.stream(
    {"messages": [{"role": "user", "content": "杭州天气怎么样？"}]},
    stream_mode="values"
):
    # 获取最新的一条消息
    if event.get("messages"):
        last_msg = event["messages"][-1]
        role = getattr(last_msg, 'type', '')
        content = getattr(last_msg, 'content', '')
        if role == "ai" and content:
            print(f"[Agent] {content}")
        elif role == "tool":
            print(f"[Tool] {getattr(last_msg, 'name', '')}: {content}")

---

## 7. 总结：v1 核心要点回顾

| 模块 | v1 最佳实践 |
|------|------------|
| **模型初始化** | `init_chat_model("openai:gpt-4o-mini")` |
| **提示词模板** | `ChatPromptTemplate.from_messages([...])` |
| **构建链** | LCEL：`prompt \| model \| StrOutputParser()` |
| **回调** | 自定义 `BaseCallbackHandler` / `UsageMetadataCallbackHandler` |
| **对话记忆** | 手动管理消息列表 / LangGraph Checkpointing |
| **Agent** | `create_agent(model=..., tools=[...], system_prompt=...)` |
| **包导入** | 优先 `from langchain.xxx import` 而非 `langchain_core` |

### v1 带来的核心收益

1. **更简洁**：`init_chat_model` 一行初始化，LCEL 替代旧式 Chain
2. **更统一**：`content_blocks` 跨供应商一致，消息 API 重新导出
3. **更强大**：`create_agent` + 中间件系统，构建复杂 Agent 更灵活
4. **更适合生产**：基于 LangGraph，天然支持持久化、流式、人机协同

### 下一步学习建议

- 📘 [LangChain v1 官方文档](https://docs.langchain.com/oss/python/)
- 📘 [create_agent 中间件指南](https://docs.langchain.com/oss/python/langchain/agents)
- 📘 [LangGraph 入门教程](https://langchain-ai.github.io/langgraph/)
- 📘 [Standard Content Blocks API](https://docs.langchain.com/oss/python/langchain/messages)

---

*本笔记对应原版 [LangChain v0.2 学习笔记](./LangChain.ipynb)，已全面升级至 v1.x API。*